# Fine-tune DistilBERT (Stage 3)

This notebook trains a **DistilBERT** text classifier on the same OCR CSV the Stage 2 TF-IDF baseline used (`data/processed/rvl_cdip_text.csv`), then evaluates on the **same test split** so the two models are comparable.

**Where to run it:** a GPU. A free Colab T4 is enough. The coding-agent environment for this repo does **not** have a GPU (PyTorch is not even installed there), so you will almost certainly run this cell-by-cell in Colab or Kaggle yourself.

**Colab setup (do this first):**
1. Runtime → Change runtime type → Hardware accelerator → **T4 GPU**.
2. Upload `data/processed/rvl_cdip_text.csv` **and** `src/evaluation/metrics.py` (Stage 2). This notebook *imports* those metrics; it does not rewrite them.
3. Run all cells. Plan on **about 15–25 minutes** on a T4, including the first-time model download.
4. Download the saved `distilbert_classifier` folder and put it at `models_store/distilbert_classifier/` in the real repo (files like `config.json` should sit *directly* in that folder).

A standalone copy of the same training job lives at `scripts/train_transformer_colab.py` if you prefer a `.py` file on a machine that already has the repo and a GPU.

In [ ]:
# Colab / Kaggle already have PyTorch. These extras pull HuggingFace Trainer + metrics.
# Skip this cell if you are running inside the repo venv with requirements.txt installed.
%pip install -q "transformers>=4.41" datasets evaluate accelerate scikit-learn pandas matplotlib seaborn

In [ ]:
from __future__ import annotations

import importlib.util
import inspect
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Same 8 classes, same order as scripts/download_data.py (Stage 2 subset).
LABELS = [
    "letter",
    "form",
    "email",
    "scientific report",
    "budget",
    "invoice",
    "resume",
    "memo",
]
LABEL2ID = {name: i for i, name in enumerate(LABELS)}
ID2LABEL = {i: name for i, name in enumerate(LABELS)}
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 512
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
SEED = 42
KNOWN_SPLITS = {"train", "validation", "val", "test"}

set_seed(SEED)

print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    BATCH_SIZE = 16  # requested ~16; DistilBERT + 512 tokens fits a T4 in fp16
else:
    print(
        "WARNING: no GPU detected. Enable a T4 in Colab or this will take hours.\n"
        "Runtime → Change runtime type → T4 GPU, then re-run from the top."
    )
    BATCH_SIZE = 4

## Paths, CSV upload, and Stage 2 metrics

Set `CSV_PATH` / `METRICS_FILE` if your files are on Google Drive instead of `/content`.

If this cell cannot find `metrics.py`, it will prompt you to upload it (Colab only).

In [ ]:
def _repo_root() -> Path:
    cwd = Path.cwd()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "src" / "evaluation" / "metrics.py").exists():
            return candidate
    return Path("/content") if IN_COLAB else cwd


ROOT = _repo_root()
CSV_PATH = ROOT / "data" / "processed" / "rvl_cdip_text.csv"
METRICS_FILE = ROOT / "src" / "evaluation" / "metrics.py"
OUTPUT_DIR = ROOT / "models_store" / "distilbert_classifier"
RESULTS_DIR = ROOT / "results"

if IN_COLAB:
    OUTPUT_DIR = Path("/content/distilbert_classifier")
    RESULTS_DIR = Path("/content/results")
    if not CSV_PATH.exists():
        CSV_PATH = Path("/content/rvl_cdip_text.csv")
    if not METRICS_FILE.exists():
        METRICS_FILE = Path("/content/metrics.py")

if IN_COLAB and not CSV_PATH.exists():
    from google.colab import files

    print("Upload data/processed/rvl_cdip_text.csv")
    uploaded = files.upload()
    CSV_PATH = Path("/content") / next(iter(uploaded))

if IN_COLAB and not METRICS_FILE.exists():
    from google.colab import files

    print("Upload src/evaluation/metrics.py (the Stage 2 file — do not substitute another copy)")
    uploaded = files.upload()
    METRICS_FILE = Path("/content") / next(iter(uploaded))


def import_stage2_metrics(metrics_file: Path):
    """Load Stage 2's compute_classification_metrics / plot_confusion_matrix."""
    src_dir = ROOT / "src"
    if src_dir.exists() and str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))
    try:
        from evaluation.metrics import (
            compute_classification_metrics,
            plot_confusion_matrix,
        )
        return compute_classification_metrics, plot_confusion_matrix
    except ImportError:
        pass
    if not metrics_file.exists():
        raise FileNotFoundError(
            "Need src/evaluation/metrics.py. Upload it; do not rewrite those functions here."
        )
    spec = importlib.util.spec_from_file_location("stage2_metrics", metrics_file)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.compute_classification_metrics, module.plot_confusion_matrix


compute_classification_metrics, plot_confusion_matrix = import_stage2_metrics(METRICS_FILE)
print("csv:", CSV_PATH, "exists:", CSV_PATH.exists())
print("metrics:", METRICS_FILE, "exists:", METRICS_FILE.exists())
print("output:", OUTPUT_DIR)

## Load CSV → HuggingFace `Dataset`

Stage 2 encoded the split in `doc_id` (`train_00000`, `test_00012`, …). We use those prefixes so train/val/test membership matches the baseline exactly. Trainer watches **validation** each epoch; the number we report for comparison is **test only**.

In [ ]:
def split_from_doc_id(doc_id: str):
    prefix = str(doc_id).split("_", 1)[0]
    return prefix if prefix in KNOWN_SPLITS else None


frame = pd.read_csv(CSV_PATH)
unknown = sorted(set(frame["label"].astype(str)) - set(LABELS))
if unknown:
    raise SystemExit(f"CSV labels not in the Stage 2 class list: {unknown}")

frame["split"] = frame["doc_id"].map(split_from_doc_id)
if not frame["split"].notna().all():
    raise SystemExit("doc_id prefixes missing; re-run Stage 2 OCR so splits stay comparable.")

train_frame = frame[frame["split"] == "train"]
val_frame = frame[frame["split"].isin(["validation", "val"])]
test_frame = frame[frame["split"] == "test"]
print(
    f"train={len(train_frame)} validation={len(val_frame)} test={len(test_frame)} "
    f"(Stage 2 scored the test split only)"
)


def frame_to_dataset(part: pd.DataFrame) -> Dataset:
    work = pd.DataFrame(
        {
            "text": part["text"].fillna("").astype(str),
            "labels": part["label"].astype(str).map(LABEL2ID).astype(int),
        }
    )
    return Dataset.from_pandas(work, preserve_index=False)


raw_datasets = DatasetDict(
    {
        "train": frame_to_dataset(train_frame),
        "validation": frame_to_dataset(val_frame),
        "test": frame_to_dataset(test_frame),
    }
)
raw_datasets

## Tokenize

`distilbert-base-uncased` does **not** read raw characters. The tokenizer:
1. Lowercases the OCR text.
2. Splits it into **subwords** from DistilBERT's ~30k WordPiece vocabulary (`invoice` might stay one token; a rare OCR garble becomes several pieces plus `[UNK]` if needed).
3. Adds `[CLS]` at the front and `[SEP]` at the end (the classification head reads `[CLS]`).
4. **Truncates** at 512 tokens (DistilBERT's maximum). Longer reports lose the tail.
5. Later, the collator **pads** shorter sequences in a batch and builds an **attention mask** (1 = real token, 0 = pad) so pad tokens do not affect the representation.

We map with `truncation=True, max_length=512` and leave padding to `DataCollatorWithPadding` so each batch only pads to its longest item, not always to 512.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)


tokenized = raw_datasets.map(tokenize_batch, batched=True, remove_columns=["text"])
print(tokenized)
print("example keys:", tokenized["train"][0].keys())
print("input_ids (first 20):", tokenized["train"][0]["input_ids"][:20])

## Model, `TrainingArguments`, and `Trainer`

We start from **pretrained** DistilBERT weights (English Wikipedia + BookCorpus), then only adapt them to our 8 document types. That is why this works on ~3,200 labeled OCR docs: the model already knows English; we are teaching it "this looks like an invoice vs a memo," not language from scratch.

| Argument | What it controls | Why this value |
|---|---|---|
| `per_device_train_batch_size=16` | How many documents update the weights together | Standard for DistilBERT on a T4; smaller if you OOM |
| `num_train_epochs=3` | Passes over the training set | Enough to adapt a pretrained encoder without frying it on a small set |
| `learning_rate=2e-5` | Step size for AdamW | Typical fine-tune LR: large enough to move the head, small enough not to erase pretrained features |
| `eval_strategy="epoch"` | When to run validation | Once per epoch is enough on this set and matches `save_strategy` |
| `fp16=True` (GPU only) | Mixed-precision math | ~2× faster and less VRAM on T4; **must stay False on CPU** |

In [ ]:
import evaluate

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(
            predictions=predictions, references=labels
        )["accuracy"],
        "f1": f1_metric.compute(
            predictions=predictions, references=labels, average="macro"
        )["f1"],
    }


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ta_kwargs = dict(
    output_dir=str(OUTPUT_DIR / "training_run"),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=torch.cuda.is_available(),
    save_strategy="epoch",
    logging_strategy="epoch",
    save_total_limit=2,
    seed=SEED,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
)
eval_key = (
    "eval_strategy"
    if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters
    else "evaluation_strategy"
)
ta_kwargs[eval_key] = "epoch"
training_args = TrainingArguments(**ta_kwargs)

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer
trainer = Trainer(**trainer_kwargs)

In [ ]:
train_result = trainer.train()
train_result

## Test-set report (same split as Stage 2) and save weights

This cell scores **test_*** rows only, writes the classification report + confusion matrix through Stage 2's `metrics.py`, and saves the model with `save_pretrained` so `src/models/transformer_classifier.py` can load it later.

In [ ]:
test_output = trainer.predict(tokenized["test"])
y_pred_ids = np.argmax(test_output.predictions, axis=-1)
y_true_ids = np.array(tokenized["test"]["labels"])
y_pred = [ID2LABEL[int(i)] for i in y_pred_ids]
y_true = [ID2LABEL[int(i)] for i in y_true_ids]

metrics = compute_classification_metrics(y_true, y_pred)
report_path = RESULTS_DIR / "transformer_classification_report.json"
report_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
plot_confusion_matrix(
    y_true,
    y_pred,
    labels=LABELS,
    save_path=RESULTS_DIR / "transformer_confusion_matrix.png",
)

trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(json.dumps(
    {
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
        "model_dir": str(OUTPUT_DIR),
        "report_path": str(report_path),
    },
    indent=2,
))

## Download the model (Colab)

Unzip into `models_store/distilbert_classifier/` in the repo. After unzipping you should see `config.json` **inside that folder**, not inside a nested extra directory. Then run:

`python scripts/compare_baseline_vs_transformer.py`

In [ ]:
if IN_COLAB:
    import shutil
    from google.colab import files

    zip_path = shutil.make_archive("/content/distilbert_classifier", "zip", OUTPUT_DIR)
    print("Downloading", zip_path)
    files.download(zip_path)
else:
    print("Saved to", OUTPUT_DIR)